# Baseline climatologica - variaveis fisicas com dados tratados

Este notebook cria uma baseline deterministica para prever irradiacao solar diaria e velocidade media do vento usando apenas historico dos dados tratados. A geracao em kWh e calculada depois da predicao fisica.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd

from src.modeling.gold_energy import (
    ENERGY_RESULT_COLUMNS,
    TARGET_COLUMNS,
    build_prediction_results_table,
    calculate_energy_outputs_from_physical,
    clip_physical_predictions,
    configure_mlflow_tracking,
    evaluate_predictions,
    load_gold_daily,
    make_future_feature_frame,
    prepare_energy_modeling_table,
    split_feature_target_metadata,
    temporal_train_test_split,
    to_jsonable,
)
from src.modeling.historical_features import (
    ClimatologyBaselineRegressor,
    fit_historical_feature_reference,
    predict_climatology_baseline,
    save_historical_reference,
)
from src.modeling.training_config import (
    BASELINE_MODEL_NAME,
    ENERGY_CONFIG,
    FUTURE_DATE,
    FUTURE_STATION_CODE,
    HISTORY_MIN_OBSERVATIONS_DAY,
    HISTORY_MIN_OBSERVATIONS_MONTH,
    MLFLOW_EXPERIMENT_NAME,
    PRODUCTION_REFIT_WITH_FULL_GOLD,
    TEST_YEAR_FRACTION,
)

RUN_ARTIFACTS_DIR = configure_mlflow_tracking(PROJECT_ROOT, MLFLOW_EXPERIMENT_NAME)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretorio de artefatos de modelagem: {RUN_ARTIFACTS_DIR}")

In [ ]:
# Configuracoes da baseline ficam em src/modeling/training_config.py.
MODEL_NAME = BASELINE_MODEL_NAME

assert HISTORY_MIN_OBSERVATIONS_DAY > 0, "HISTORY_MIN_OBSERVATIONS_DAY deve ser positivo."
assert HISTORY_MIN_OBSERVATIONS_MONTH > 0, "HISTORY_MIN_OBSERVATIONS_MONTH deve ser positivo."
print(f"HISTORY_MIN_OBSERVATIONS_DAY={HISTORY_MIN_OBSERVATIONS_DAY}")
print(f"HISTORY_MIN_OBSERVATIONS_MONTH={HISTORY_MIN_OBSERVATIONS_MONTH}")
print(f"Refit de producao com todos os dados tratados={PRODUCTION_REFIT_WITH_FULL_GOLD}")

In [ ]:
daily_gold = load_gold_daily(PROJECT_ROOT)
modeling_table = prepare_energy_modeling_table(daily_gold, ENERGY_CONFIG)
X, y, metadata = split_feature_target_metadata(modeling_table)

(
    X_train,
    X_test,
    y_train,
    y_test,
    train_metadata,
    test_metadata,
    train_years,
    test_years,
) = temporal_train_test_split(X, y, metadata, test_year_fraction=TEST_YEAR_FRACTION)

baseline_model = ClimatologyBaselineRegressor(
    min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
    min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
)
X_train_for_baseline = X_train.copy()
X_train_for_baseline["year"] = train_metadata["year"].to_numpy()
baseline_model.fit(X_train_for_baseline, y_train)
train_reference = baseline_model.reference_
production_reference = (
    fit_historical_feature_reference(
        modeling_table,
        min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
        min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
    )
    if PRODUCTION_REFIT_WITH_FULL_GOLD
    else train_reference
)

test_predictions = clip_physical_predictions(baseline_model.predict(X_test))
metrics_table, final_metrics = evaluate_predictions(y_test, test_predictions)
test_results, energy_actual, energy_pred, _ = build_prediction_results_table(
    test_metadata,
    y_test,
    test_predictions,
    ENERGY_CONFIG,
)
energy_metrics_table, energy_metrics = evaluate_predictions(
    energy_actual[ENERGY_RESULT_COLUMNS],
    energy_pred[ENERGY_RESULT_COLUMNS],
    target_columns=ENERGY_RESULT_COLUMNS,
)

print(f"Linhas de dados tratados usadas: {len(modeling_table):,}")
print(f"Alvos fisicos do ML: {TARGET_COLUMNS}")
print(f"Saidas energeticas calculadas: {ENERGY_RESULT_COLUMNS}")
print(f"Anos treino/validacao: {train_years}")
print(f"Anos teste final: {test_years}")
print("Metricas fisicas da baseline no teste temporal final:")
display(metrics_table)
print("Metricas energeticas calculadas da baseline:")
display(energy_metrics_table)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"{MODEL_NAME}_physical_gold"):
    mlflow_run_id = mlflow.active_run().info.run_id
    mlflow.log_param("model_family", "ClimatologyBaselineRegressor")
    mlflow.log_param("train_years", ",".join(map(str, train_years)))
    mlflow.log_param("test_years", ",".join(map(str, test_years)))
    mlflow.log_param("targets", ",".join(TARGET_COLUMNS))
    mlflow.log_param("energy_outputs", ",".join(ENERGY_RESULT_COLUMNS))
    mlflow.log_param("history_min_observations_day", HISTORY_MIN_OBSERVATIONS_DAY)
    mlflow.log_param("history_min_observations_month", HISTORY_MIN_OBSERVATIONS_MONTH)
    mlflow.log_param("production_refit_with_full_gold", PRODUCTION_REFIT_WITH_FULL_GOLD)
    mlflow.log_dict(to_jsonable(ENERGY_CONFIG), "energy_config.json")
    mlflow.log_dict(to_jsonable(train_reference.metadata()), "historical_reference_train_metadata.json")
    mlflow.log_dict(to_jsonable(production_reference.metadata()), "historical_reference_production_metadata.json")
    for metric_name, metric_value in final_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"physical_{metric_name}", float(metric_value))
    for metric_name, metric_value in energy_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"energy_{metric_name}", float(metric_value))
    mlflow.sklearn.log_model(baseline_model, artifact_path="model")

print("RESULTADOS FINAIS - BASELINE CLIMATOLOGICA")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de treino/validacao: {train_years}")
print(f"Anos de teste final: {test_years}")
display(metrics_table)
display(energy_metrics_table)

In [ ]:
# Resumo explicito das metricas fisicas e energeticas no teste temporal final.
print("RESUMO DAS METRICAS - TESTE TEMPORAL FINAL")
print(f"Modelo: {MODEL_NAME}")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de teste final: {test_years}")
print(f"physical_balanced_nrmse: {final_metrics['balanced_nrmse']:.6f}")

print("")
print("Metricas fisicas previstas pelo ML:")
for _, row in metrics_table.iterrows():
    print(f"Alvo: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")
print("")
print("Metricas energeticas calculadas apos a predicao fisica:")
for _, row in energy_metrics_table.iterrows():
    print(f"Saida: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")

In [ ]:
results_dir = RUN_ARTIFACTS_DIR / "evaluation" / MODEL_NAME
results_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
metrics_output_path = results_dir / f"{MODEL_NAME}_physical_metrics_{run_timestamp}.csv"
energy_metrics_output_path = results_dir / f"{MODEL_NAME}_energy_metrics_{run_timestamp}.csv"
metrics_json_output_path = results_dir / f"{MODEL_NAME}_physical_metrics_{run_timestamp}.json"
energy_metrics_json_output_path = results_dir / f"{MODEL_NAME}_energy_metrics_{run_timestamp}.json"
predictions_output_path = results_dir / f"{MODEL_NAME}_test_predictions_{run_timestamp}.csv"
predictions_sample_output_path = results_dir / f"{MODEL_NAME}_test_predictions_sample_{run_timestamp}.csv"
train_reference_dir = results_dir / "historical_reference_train"
production_reference_dir = results_dir / "historical_reference_production"

metrics_table.to_csv(metrics_output_path, index=False)
energy_metrics_table.to_csv(energy_metrics_output_path, index=False)
pd.Series(final_metrics).to_json(metrics_json_output_path, indent=2)
pd.Series(energy_metrics).to_json(energy_metrics_json_output_path, indent=2)
test_results.to_csv(predictions_output_path, index=False)
test_results.head(50).to_csv(predictions_sample_output_path, index=False)
save_historical_reference(train_reference, train_reference_dir)
save_historical_reference(production_reference, production_reference_dir)

with mlflow.start_run(run_id=mlflow_run_id):
    mlflow.log_artifact(str(metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(metrics_json_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_metrics_json_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(predictions_sample_output_path), artifact_path="evaluation")
    mlflow.log_artifacts(str(train_reference_dir), artifact_path="historical_reference_train")
    mlflow.log_artifacts(str(production_reference_dir), artifact_path="historical_reference_production")

print("Arquivos de resultados salvos:")
print(f"- Metricas fisicas CSV: {metrics_output_path}")
print(f"- Metricas energeticas CSV: {energy_metrics_output_path}")
print(f"- Predicoes completas do teste: {predictions_output_path}")
print(f"- Amostra das predicoes: {predictions_sample_output_path}")
print(f"- Referencia historica de treino: {train_reference_dir}")
print(f"- Referencia historica de producao: {production_reference_dir}")
print("Primeiras predicoes do teste temporal:")
display(test_results.head(20))

In [ ]:
# Preencha FUTURE_STATION_CODE e FUTURE_DATE em src/modeling/training_config.py.
if FUTURE_STATION_CODE is None or FUTURE_DATE is None:
    print("Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.")
else:
    future_frame = make_future_feature_frame(modeling_table, FUTURE_DATE)
    future_predictions = clip_physical_predictions(predict_climatology_baseline(future_frame, production_reference))
    future_energy = calculate_energy_outputs_from_physical(future_predictions, ENERGY_CONFIG, include_hub_height=True)
    future_ranking = future_frame[[
        column
        for column in ["station_code", "station_name", "city", "state", "latitude", "longitude", "altitude", "date"]
        if column in future_frame.columns
    ]].copy()
    for column in TARGET_COLUMNS:
        future_ranking[f"{column}_pred"] = future_predictions[column].to_numpy()
    for column in future_energy.columns:
        future_ranking[f"{column}_pred"] = future_energy[column].to_numpy()
    future_ranking = future_ranking.sort_values("hybrid_generation_kwh_day_pred", ascending=False).reset_index(drop=True)

    station_code = str(FUTURE_STATION_CODE)
    station_prediction = future_ranking[future_ranking["station_code"].astype(str) == station_code].copy()
    if station_prediction.empty:
        known = ", ".join(future_ranking["station_code"].astype(str).head(10).tolist())
        raise ValueError(f"station_code nao encontrado nos dados tratados: {station_code}. Exemplos conhecidos: {known}")
    station_prediction.insert(0, "ranking_position", station_prediction.index + 1)
    display(station_prediction.reset_index(drop=True))
    display(future_ranking.head(20))